# LLM Quantization for Efficient Inference

## Overview
Large Language Models (LLMs) are computationally expensive and memory-intensive. **Quantization** reduces model size and memory usage by representing weights with fewer bits, enabling efficient inference on resource-constrained devices.

## Quantization Techniques Covered
- **Full Precision (FP32)**: 32-bit floating point
- **Half Precision (BF16/FP16)**: 16-bit floating point  
- **INT8 Quantization**: 8-bit integer representation
- **4-bit Quantization**: QLoRA with NF4 format
- **GPTQ**: Post-training quantization

> **⚠️ GPU Required**: This notebook requires CUDA-compatible GPU for quantization operations. 

## Model Selection

In [ ]:
# Using Llama-3.2-1B-Instruct as our test model
model_id = "meta-llama/Llama-3.2-1B-Instruct"

## 1. Full Precision (FP32)

**32-bit floating point** - Standard precision used in most deep learning models.
- **Memory**: ~4.6 GB for 1B parameter model
- **Accuracy**: Highest precision, no quantization loss
- **Use case**: Training and high-accuracy inference

In [ ]:
# Load model in full precision (FP32)
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    trust_remote_code=True, 
    device_map="auto"  # Automatically place model on available GPU
)

In [ ]:
# Check model precision and memory usage
print(f"Model dtype: {model.dtype}")
print(f"GPU memory: {model.get_memory_footprint() / 1024**3:.2f} GB")

In [ ]:
# Display model architecture
print(model)

**Key Observations:**
- Linear layers use standard `Linear` modules
- Weights stored in FP32 precision
- All computations performed in FP32

In [ ]:
# Clean up memory before loading next model
import torch

del model
torch.cuda.empty_cache()  # Clear GPU memory

## 2. Half Precision (BF16/FP16)

**16-bit floating point** - Reduces memory usage by ~50% with minimal accuracy loss.
- **BF16**: Better numerical stability, wider dynamic range
- **FP16**: Standard half precision, may cause overflow issues
- **Memory**: ~2.3 GB for 1B parameter model
- **Use case**: Inference with good speed/accuracy trade-off

In [ ]:
# Load model in BF16 precision
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    dtype=torch.bfloat16,  # Use BF16 for better numerical stability
    trust_remote_code=True, 
    device_map="auto"
)

In [ ]:
# Check memory reduction
print(f"Model dtype: {model.dtype}")
print(f"GPU memory: {model.get_memory_footprint() / 1024**3:.2f} GB")

**Memory Reduction**: ~50% reduction from FP32 (4.6 GB → 2.3 GB)

In [ ]:
# Display model architecture
print(model)

**Key Observations:**
- Weights stored in BF16 precision
- All computations performed in BF16
- Same architecture as FP32 model

In [ ]:
# Clean up memory
del model
torch.cuda.empty_cache()

## 3. INT8 Quantization

**8-bit integer representation** - Further reduces memory with minimal accuracy loss.
- **Memory**: ~1.15 GB for 1B parameter model (75% reduction)
- **Method**: Uses bitsandbytes library for efficient 8-bit operations
- **Use case**: Memory-constrained inference

### Method 1: Using Transformers Parameters

In [ ]:
# Load model with 8-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    dtype=torch.bfloat16,  # Computation dtype
    trust_remote_code=True, 
    device_map="auto",
    load_in_8bit=True  # Enable 8-bit quantization
)

In [ ]:
# Check memory usage with 8-bit quantization
print(f"Model dtype: {model.dtype}")
print(f"GPU memory: {model.get_memory_footprint() / 1024**3:.2f} GB")

**Key Observations:**
- 75% memory reduction from FP32 (4.6 GB → 1.15 GB)
- Weights stored in 8-bit, computations in BF16
- Linear layers replaced with `Linear8bitLt` modules

In [ ]:
# Display quantized model architecture
print(model)

Simple Linear is replaced with  **Linear8bitLt**.

In [ ]:
del model
torch.cuda.empty_cache()

Model `dtype` shows `torch.bfloat16`: This means the model will store weigths with **8bit**. While the computation will happen in `torch.bfloat16`.

### Using bitsandbytes quantization config

In [ ]:
from transformers import BitsAndBytesConfig

In [ ]:
quantization_config = BitsAndBytesConfig(load_in_8bit=True,
                                         llm_int8_threshold=200.0)

model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, trust_remote_code=True, device_map="auto",
    quantization_config=quantization_config
)

In [ ]:
print(model.dtype)
print(f"GPU memory: {model.get_memory_footprint() / 1024**3:.2f} GB")

In [ ]:
print(model)

In [ ]:
del model
torch.cuda.empty_cache()

## 4bit Model Quantization

### Using transformers parameters

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.bfloat16, trust_remote_code=True, device_map="auto",
            load_in_4bit=True
        )

In [ ]:
print(model.dtype)
print(f"GPU memory: {model.get_memory_footprint() / 1024**3:.2f} GB")

* Model `dtype` shows `torch.bfloat16`: This means the model will store weigths with **4bit**. While the computation will happen in `torch.bfloat16`.
* Model size further reduced.

In [ ]:
print(model)

Simple Linear is replaced with  **Linear4bit**.

In [ ]:
del model
torch.cuda.empty_cache()

### using bitsandbytes quantization config

#### QLoRA NF4

In [ ]:
bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16
        )
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, trust_remote_code=True, device_map="auto"
)

In [ ]:
print(model.dtype)
print(f"GPU memory: {model.get_memory_footprint() / 1024**3:.2f} GB")
print(f"GPU memory: {model.get_memory_footprint():.2f} B")

In [ ]:
print(model)

In [ ]:
del model
torch.cuda.empty_cache()

#### QLoRA NF4-double-quantization (Nested Quantization)

Using `bnb_4bit_use_double_quant` argument, double or nested quantization can be achieved. This will enable second quantization after the first one to save additional 0.4 bits per parameter.

In [ ]:
bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, trust_remote_code=True, device_map="auto"
)

In [ ]:
print(model.dtype)
print(f"GPU memory: {model.get_memory_footprint() / 1024**3:.2f} GB")
print(f"GPU memory: {model.get_memory_footprint():.2f} B")

Here, the impact of double quantization is not clearly evident. However, for larger models, there is an observable difference.

In [ ]:
print(model)

In [ ]:
del model
torch.cuda.empty_cache()

## 5. GPTQ (Post-Training Quantization)

**GPTQ** - Post-training quantization with calibration dataset.
- **Memory**: ~0.6 GB for 1B parameter model (87% reduction)
- **Method**: Calibrated quantization using representative data
- **Use case**: High accuracy with maximum compression

In [ ]:
# Import required libraries for GPTQ quantization
from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig
import torch

In [ ]:
# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)

**GPTQ Calibration Process:**
- Requires representative dataset for calibration
- Supported datasets: `wikitext2`, `c4`, `ptb`, etc.
- Can use custom dataset as list of strings
- **Note**: Quantization process takes several hours

In [ ]:
# Configure GPTQ quantization settings
quantization_config = GPTQConfig(
    bits=4,              # 4-bit quantization
    group_size=128,      # Group size for quantization
    dataset="c4",        # Calibration dataset
    desc_act=False,      # Disable desc_act for better compatibility
    tokenizer=tokenizer  # Required for text processing
)

### Loading the model and quantizing

In [ ]:
# Load the model from HF
quant_model = AutoModelForCausalLM.from_pretrained(model_id, 
                quantization_config=quantization_config, trust_remote_code=True, device_map='auto')

This process takes few hours to generate a quantized model. Hence, it is recommended to save the model locally or push it to huggingface_hub for later use.

### Saving the quantized model and tokenizer

In [ ]:
# save the quantize model to disk
save_folder = "./models/quantized-shearedllama-2.7b"
quant_model.save_pretrained(save_folder, safe_serialization=True)

In [ ]:
# save the tokenizer
tokenizer.save_pretrained(save_folder)

In [ ]:
del quant_model
torch.cuda.empty_cache()

### Loading the Quantized Model

In [ ]:
model_path = "./models/quantized-shearedllama-2.7b/"
gptq_config = GPTQConfig(bits=4, disable_exllama=False)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto", 
                                             quantization_config = gptq_config)

In [ ]:
print(model.dtype)
print(f"GPU memory: {model.get_memory_footprint() / 1024**3:.2f} GB")

In [ ]:
print(model)

Linear layer will be modified by by **QuantLinear** layer from auto-gptq.

In [ ]:
print(model.config.quantization_config.to_dict())

In [ ]:
del model
torch.cuda.empty_cache()

## Imprtant References
* HuggingFace Blogs
    1. https://huggingface.co/blog/hf-bitsandbytes-integration
    2. https://huggingface.co/blog/4bit-transformers-bitsandbytes
    3. https://huggingface.co/blog/gptq-integration
    4. https://huggingface.co/blog/merve/quantization
    5. https://huggingface.co/blog/overview-quantization-transformers
    6. https://huggingface.co/docs/transformers/v4.34.1/en/main_classes/quantization
* Research Papers
    1. [LLM.int8](https://arxiv.org/abs/2208.07339)
    2. [QLoRA](https://arxiv.org/abs/2305.14314)
    3. [GPTQ](https://arxiv.org/pdf/2210.17323.pdf)
* Github Repo
    1. [bitsandbytes](https://github.com/TimDettmers/bitsandbytes)
    2. [auto-gptq](https://github.com/PanQiWei/AutoGPTQ)